In [1]:
import numpy as np
import pandas as pd
import torch
import folium
from IPython.display import display

from lstm import lstm_seq2seq

In [ ]:
# =========================
# CONFIG
# =========================
TRAIN_CSV = "../data/vessel_prediction_model_data/mid_model_data/mid_train_data.csv"
VAL_CSV   = "../data/vessel_prediction_model_data/mid_model_data/mid_val_data.csv"
TEST_CSV  = ".../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv"


SEQ_LEN = 10
TARGET_LEN = 1

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
RANDOM_STATE = 42

BATCH_SIZE = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
MAX_EPOCHS = 100
TEACHER_FORCING_RATIO = 0.7

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = "best_lstm_seq2seq_dlat_dlon.pt"

feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]
route_col = "voyage_id"
time_col = "TIME"

YARDS_PER_METER = 1.0936132983377078

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df_raw = train_df.copy()
val_df_raw = val_df.copy()
test_df_raw = test_df.copy()

In [4]:
# =========================
# SCALE FROM TRAIN ONLY
# =========================
feature_means = train_df_raw[feature_cols].mean()
feature_stds = train_df_raw[feature_cols].std().replace(0, 1.0)

target_means = train_df_raw[target_cols].mean()
target_stds = train_df_raw[target_cols].std().replace(0, 1.0)

train_df = train_df_raw.copy()
val_df = val_df_raw.copy()
test_df = test_df_raw.copy()

train_df[feature_cols] = (train_df[feature_cols] - feature_means) / feature_stds
val_df[feature_cols] = (val_df[feature_cols] - feature_means) / feature_stds
test_df[feature_cols] = (test_df[feature_cols] - feature_means) / feature_stds

train_df[target_cols] = (train_df[target_cols] - target_means) / target_stds
val_df[target_cols] = (val_df[target_cols] - target_means) / target_stds
test_df[target_cols] = (test_df[target_cols] - target_means) / target_stds

In [5]:
# =========================
# WINDOWING BY VOYAGE_ID
# =========================
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len,
    target_len,
    route_col="voyage_id",
    time_col="TIME",
):
    """
    Returns
    -------
    X : torch.FloatTensor
        Shape (seq_len, N, n_features)
    Y : torch.FloatTensor
        Shape (target_len, N, n_targets)
    meta_df : pd.DataFrame
        One row per window with route/sample metadata
    """
    X_list = []
    Y_list = []
    meta_rows = []

    work = df.copy()
    work[time_col] = pd.to_datetime(work[time_col])

    for route_id, g in work.groupby(route_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)
        raw_latlon = g[["LAT", "LON"]].to_numpy(dtype=np.float64)
        raw_times = g[time_col].to_numpy()

        n = len(g)
        if n < seq_len + target_len:
            continue

        for start in range(n - (seq_len + target_len) + 1):
            hist_end = start + seq_len
            fut_end = hist_end + target_len

            X_list.append(feats[start:hist_end])
            Y_list.append(targs[hist_end:fut_end])

            meta_rows.append(
                {
                    "voyage_id": route_id,
                    "window_start": start,
                    "history_start_time": raw_times[start],
                    "history_end_time": raw_times[hist_end - 1],
                    "target_time": raw_times[hist_end],
                    "history_latlon": raw_latlon[start:hist_end].copy(),
                    "true_future_latlon": raw_latlon[hist_end:fut_end].copy(),
                }
            )

    if not X_list:
        raise ValueError("No windows created. Reduce seq_len/target_len or inspect the data.")

    X = torch.from_numpy(np.stack(X_list, axis=0)).permute(1, 0, 2).contiguous()
    Y = torch.from_numpy(np.stack(Y_list, axis=0)).permute(1, 0, 2).contiguous()
    meta_df = pd.DataFrame(meta_rows)

    return X, Y, meta_df


X_train, Y_train, meta_train = make_windows_from_df(
    train_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)
X_val, Y_val, meta_val = make_windows_from_df(
    val_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)
X_test, Y_test, meta_test = make_windows_from_df(
    test_df, feature_cols, target_cols, seq_len=SEQ_LEN, target_len=TARGET_LEN,
    route_col=route_col, time_col=time_col
)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_val:  ", X_val.shape, "Y_val:  ", Y_val.shape)
print("X_test: ", X_test.shape, "Y_test: ", Y_test.shape)
print("\nTest windows by voyage (head):")
display(meta_test[["voyage_id", "window_start", "history_end_time", "target_time"]].head())

X_train: torch.Size([10, 14696, 10]) Y_train: torch.Size([1, 14696, 2])
X_val:   torch.Size([10, 4182, 10]) Y_val:   torch.Size([1, 4182, 2])
X_test:  torch.Size([10, 4581, 10]) Y_test:  torch.Size([1, 4581, 2])

Test windows by voyage (head):


,voyage_id,window_start,history_end_time,target_time
0,72_209425000,0,2024-02-13 13:50:00,2024-02-13 13:55:00
1,72_209425000,1,2024-02-13 13:55:00,2024-02-13 14:00:00
2,72_209425000,2,2024-02-13 14:00:00,2024-02-13 14:05:00
3,72_209425000,3,2024-02-13 14:05:00,2024-02-13 14:15:00
4,72_209425000,4,2024-02-13 14:15:00,2024-02-13 14:20:00


In [6]:
# =========================
# MODEL + TRAINING
# =========================
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    target_indices=(0, 1),  # dlat, dlon in target tensor
    decoder_feature_indices=tuple(range(len(feature_cols))),
    predict_deltas=True,
).to(DEVICE)

history = model.train_model(
    input_tensor=X_train,
    target_tensor=Y_train,
    target_len=TARGET_LEN,
    batch_size=BATCH_SIZE,
    val_input_tensor=X_val,
    val_target_tensor=Y_val,
    max_epochs=MAX_EPOCHS,
    training_prediction="mixed_teacher_forcing",
    teacher_forcing_ratio=TEACHER_FORCING_RATIO,
    learning_rate=LEARNING_RATE,
    dynamic_tf=True,
    patience=10,
    min_delta=1e-5,
    save_path=SAVE_PATH,
    grad_clip=1.0,
    shuffle_batches=True,
)

print("Training done.")
print("Best epoch:", history["best_epoch"])
print("Best val loss:", history["best_val_loss"])

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

 17%|█▋        | 17/100 [00:24<01:57,  1.42s/it, no_improve=10, train=196.9215, val=381.2365]

Training done.
Best epoch: 7
Best val loss: 376.71076692595625



C:\Users\logan\AppData\Local\Temp\ipykernel_45128\2076216690.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(SAVE_PATH, map_location=D

lstm_seq2seq(
  (encoder): lstm_encoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
  )
  (decoder): lstm_decoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
    (linear): Linear(in_features=128, out_features=2, bias=True)
  )
  (target_to_decoder): Linear(in_features=2, out_features=10, bias=True)
)

In [7]:
# =========================
# PREDICTION HELPERS
# =========================
def predict_dataset_deltas(model, X_tensor, target_len, batch_size=256):
    """
    Returns scaled predicted deltas with shape (target_len, N, 2).
    """
    model.eval()
    preds = []
    n_samples = X_tensor.shape[1]

    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            x_batch = X_tensor[:, start:end, :].to(DEVICE)

            _, encoder_hidden = model.encoder(x_batch)
            decoder_input = model._make_initial_decoder_input(x_batch)
            outputs = model._decode_rollout_recursive(decoder_input, encoder_hidden, target_len, x_batch)
            preds.append(outputs.detach().cpu())

    return torch.cat(preds, dim=1)


def deltas_to_absolute(history_last_abs, delta_arr):
    """
    history_last_abs: (N, 2)
    delta_arr: (target_len, N, 2)
    returns: (target_len, N, 2)
    """
    current = history_last_abs.astype(np.float64).copy()
    abs_list = []

    for t in range(delta_arr.shape[0]):
        current = current + delta_arr[t]
        abs_list.append(current.copy())

    return np.stack(abs_list, axis=0)


def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c


def metrics_yards(pred_abs, true_abs):
    """
    pred_abs, true_abs: numpy arrays of shape (target_len, N, 2)
    Computes point-to-point geodesic error in yards.
    """
    err_m = haversine_m(
        true_abs[..., 0],
        true_abs[..., 1],
        pred_abs[..., 0],
        pred_abs[..., 1],
    )
    err_yd = err_m * YARDS_PER_METER

    mae_yd = np.mean(np.abs(err_yd))
    mse_yd2 = np.mean(err_yd ** 2)
    rmse_yd = np.sqrt(mse_yd2)

    return mae_yd, mse_yd2, rmse_yd, err_yd

In [8]:
X_test_np = X_test.detach().cpu().numpy().copy()

history_abs_all = X_test_np[:, :, [0, 1]].copy()   # (seq_len, N, 2)

history_abs_all[:, :, 0] = history_abs_all[:, :, 0] * feature_stds["LAT"] + feature_means["LAT"]
history_abs_all[:, :, 1] = history_abs_all[:, :, 1] * feature_stds["LON"] + feature_means["LON"]

history_last_abs = history_abs_all[-1, :, :]       # (N, 2)

print("history_abs_all shape:", history_abs_all.shape)
print("history last abs:", history_last_abs[0])

history_abs_all shape: (10, 4581, 2)
history last abs: [ 21.78906  -76.962166]


In [9]:
# =========================
# TEST SET METRICS IN YARDS
# =========================
Y_pred_scaled = predict_dataset_deltas(model, X_test, target_len=TARGET_LEN, batch_size=256).numpy()

# Unscale predicted and true deltas
Y_pred_delta = Y_pred_scaled.copy()
Y_true_delta = Y_test.cpu().numpy().copy()

Y_pred_delta[..., 0] = Y_pred_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_pred_delta[..., 1] = Y_pred_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

Y_true_delta[..., 0] = Y_true_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_true_delta[..., 1] = Y_true_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

# =========================
# CONVERT TRUE/PRED DELTAS TO ABSOLUTE LAT/LON
# =========================

# true deltas -> numpy
Y_true_delta = Y_test.detach().cpu().numpy().copy()

# predicted deltas -> numpy
if isinstance(Y_pred_delta, torch.Tensor):
    Y_pred_delta = Y_pred_delta.detach().cpu().numpy()
else:
    Y_pred_delta = Y_pred_delta.copy()

# history_abs_all should be (seq_len, N, 2)
history_last_abs = history_abs_all[-1, :, :]   # correct for (seq_len, N, 2)

# unscale true deltas
Y_true_delta[..., 0] = Y_true_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_true_delta[..., 1] = Y_true_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

# unscale predicted deltas
Y_pred_delta[..., 0] = Y_pred_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_pred_delta[..., 1] = Y_pred_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

# convert deltas to absolute lat/lon
true_future_abs = deltas_to_absolute(history_last_abs, Y_true_delta)
pred_future_abs = deltas_to_absolute(history_last_abs, Y_pred_delta)

print("history last abs:", history_last_abs[0])
print("true future abs:", true_future_abs[0, 0])
print("pred future abs:", pred_future_abs[0, 0])

mae_yd, mse_yd2, rmse_yd, err_yd = metrics_yards(pred_future_abs, true_future_abs)

print("Test metrics in yards")
print(f"MAE  (yd): {mae_yd:,.3f}")
print(f"MSE  (yd²): {mse_yd2:,.3f}")
print(f"RMSE (yd): {rmse_yd:,.3f}")

history last abs: [ 21.78906  -76.962166]
true future abs: [ 22.31892288 -77.69736099]
pred future abs: [ 21.8053155  -76.98816859]
Test metrics in yards
MAE  (yd): 66,117.745
MSE  (yd²): 6,191,442,484.072
RMSE (yd): 78,685.720


In [10]:
YARDS_PER_METER = 1.0936132983377078

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c

def metrics_yards(pred_abs, true_abs):
    err_m = haversine_m(
        true_abs[..., 0],
        true_abs[..., 1],
        pred_abs[..., 0],
        pred_abs[..., 1],
    )
    err_yd = err_m * YARDS_PER_METER

    mae_yd = np.mean(np.abs(err_yd))
    mse_yd2 = np.mean(err_yd ** 2)
    rmse_yd = np.sqrt(mse_yd2)

    return mae_yd, mse_yd2, rmse_yd, err_yd

def dead_reckoning_next_point(lat_deg, lon_deg, speed_knots, cog_deg, dt_seconds):
    """
    Predict one next AIS point assuming constant speed and constant course.
    Returns predicted (lat, lon) in degrees.
    """
    # distance traveled in meters
    dist_m = speed_knots * 0.514444 * dt_seconds

    # bearings: 0° = north, 90° = east
    theta = np.radians(cog_deg)

    # north/east displacement
    d_north = dist_m * np.cos(theta)
    d_east  = dist_m * np.sin(theta)

    # convert meters to degrees
    dlat = d_north / 111320.0
    dlon = d_east / (111320.0 * np.cos(np.radians(lat_deg)))

    pred_lat = lat_deg + dlat
    pred_lon = lon_deg + dlon
    return pred_lat, pred_lon

In [11]:
# =========================
# DEAD RECKONING BASELINE ON TEST SET
# =========================

# convert test tensors to numpy
X_test_np = X_test.detach().cpu().numpy().copy()   # shape likely (seq_len, N, features)
Y_test_np = Y_test.detach().cpu().numpy().copy()   # shape likely (target_len, N, 2)

# feature index lookup
idx_lat = feature_cols.index("LAT")
idx_lon = feature_cols.index("LON")
idx_speed = feature_cols.index("SPEED")
idx_dt = feature_cols.index("dt")
idx_cog_sin = feature_cols.index("cog_sin")
idx_cog_cos = feature_cols.index("cog_cos")

# rebuild absolute history features from scaled X_test
X_test_unscaled = X_test_np.copy()

for col in feature_cols:
    j = feature_cols.index(col)
    X_test_unscaled[:, :, j] = X_test_unscaled[:, :, j] * feature_stds[col] + feature_means[col]

# rebuild true dlat/dlon from scaled Y_test
Y_true_delta = Y_test_np.copy()
Y_true_delta[..., 0] = Y_true_delta[..., 0] * target_stds["dlat"] + target_means["dlat"]
Y_true_delta[..., 1] = Y_true_delta[..., 1] * target_stds["dlon"] + target_means["dlon"]

# last history point for each sample
last_lat = X_test_unscaled[-1, :, idx_lat]
last_lon = X_test_unscaled[-1, :, idx_lon]
last_speed = X_test_unscaled[-1, :, idx_speed]
last_dt = X_test_unscaled[-1, :, idx_dt]
last_cog_sin = X_test_unscaled[-1, :, idx_cog_sin]
last_cog_cos = X_test_unscaled[-1, :, idx_cog_cos]

# course in degrees from sin/cos
last_cog_rad = np.arctan2(last_cog_sin, last_cog_cos)
last_cog_deg = (np.degrees(last_cog_rad) + 360.0) % 360.0

# true absolute next point from true delta
true_future_abs = np.zeros((1, X_test_unscaled.shape[1], 2), dtype=np.float64)
true_future_abs[0, :, 0] = last_lat + Y_true_delta[0, :, 0]
true_future_abs[0, :, 1] = last_lon + Y_true_delta[0, :, 1]

# dead reckoning predicted next point
pred_future_abs_dr = np.zeros((1, X_test_unscaled.shape[1], 2), dtype=np.float64)

for i in range(X_test_unscaled.shape[1]):
    pred_lat, pred_lon = dead_reckoning_next_point(
        lat_deg=last_lat[i],
        lon_deg=last_lon[i],
        speed_knots=last_speed[i],
        cog_deg=last_cog_deg[i],
        dt_seconds=last_dt[i],
    )
    pred_future_abs_dr[0, i, 0] = pred_lat
    pred_future_abs_dr[0, i, 1] = pred_lon

# metrics in yards
mae_yd_dr, mse_yd2_dr, rmse_yd_dr, err_yd_dr = metrics_yards(pred_future_abs_dr, true_future_abs)

print("Dead Reckoning baseline metrics in yards")
print(f"MAE  (yd): {mae_yd_dr:,.3f}")
print(f"MSE  (yd²): {mse_yd2_dr:,.3f}")
print(f"RMSE (yd): {rmse_yd_dr:,.3f}")

Dead Reckoning baseline metrics in yards
MAE  (yd): 66,273.233
MSE  (yd²): 6,211,913,268.406
RMSE (yd): 78,815.692


In [12]:
# =========================
# MAP FUNCTION
# =========================
def mapmake(history, prediction):
    m = folium.Map(location=[22.5, -77.8], zoom_start=8)

    history_coords = [(row[0], row[1]) for row in history]
    pred_coords = [(row[0], row[1]) for row in prediction]
    print(pred_coords)

    folium.PolyLine(history_coords, weight=3, tooltip="History").add_to(m)

    folium.Marker(
        location=list(history_coords[-1]),
        popup="Last ping",
        tooltip="Last ping"
    ).add_to(m)

    folium.PolyLine(
        pred_coords,
        weight=3,
        dash_array="5, 8",
        tooltip="Predicted"
    ).add_to(m)

    for i, (lat, lon) in enumerate(history_coords):
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            popup=f"History {i}",
            fill=True
        ).add_to(m)

    for i, (lat, lon) in enumerate(pred_coords, start=1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Predicted {i}",
            tooltip=f"Predicted {i}"
        ).add_to(m)

    return m

In [13]:
unique_test_voyages = meta_test["voyage_id"].drop_duplicates().tolist()[:3]

for map_num, voyage_id in enumerate(unique_test_voyages, start=1):
    row = meta_test[meta_test["voyage_id"] == voyage_id].iloc[0]
    sample_idx = row.name

    history_abs = history_abs_all[:, sample_idx, :]     # (seq_len, 2)
    pred_point_abs = pred_future_abs[:, sample_idx, :]  # (1, 2)
    true_point_abs = true_future_abs[:, sample_idx, :]  # (1, 2)

    print(f"Map {map_num} | voyage_id={voyage_id}")
    print("History last AIS point:  ", history_abs[-1])
    print("Predicted next AIS point:", pred_point_abs[0])
    print("True next AIS point:     ", true_point_abs[0])

    display(mapmake(history_abs, pred_point_abs))
    print("-" * 70)

Map 1 | voyage_id=72_209425000
History last AIS point:   [ 21.78906  -76.962166]
Predicted next AIS point: [ 21.8053155  -76.98816859]
True next AIS point:      [ 22.31892204 -77.69735718]
[(np.float64(21.8053154964), np.float64(-76.98816859163344))]


----------------------------------------------------------------------
Map 2 | voyage_id=85_209425000
History last AIS point:   [ 22.204374 -77.498276]
Predicted next AIS point: [ 22.22068497 -77.51828504]
True next AIS point:      [ 22.89681053 -78.03179169]
[(np.float64(22.220684967935085), np.float64(-77.51828504167497))]


----------------------------------------------------------------------
Map 3 | voyage_id=90_209425000
History last AIS point:   [ 22.234013 -77.59287 ]
Predicted next AIS point: [ 22.21837776 -77.57633796]
True next AIS point:      [ 21.77487183 -77.27851105]
[(np.float64(22.218377763405442), np.float64(-77.57633795961738))]


----------------------------------------------------------------------
